In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, roc_curve, confusion_matrix)
import os

INPUT_FILE = "lei_549300FGXN1K3HLB1R50.csv"   # raw HMDA LAR file
CHART_DIR = "charts"
os.makedirs(CHART_DIR, exist_ok=True)

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 130
COLOR_APPROVED = "#2E5C8A"
COLOR_DENIED = "#B0B0B0"

# ------------------------------------------------------------------
# 1. LOAD AND CLEAN
# ------------------------------------------------------------------
df = pd.read_csv(INPUT_FILE, low_memory=False)
print(f"Raw records: {len(df):,}")

sub = df[df["action_taken"].isin([1, 3])].copy()
sub["approved"] = (sub["action_taken"] == 1).astype(int)
print(f"Decisioned records (Originated/Denied): {len(sub):,}")
print(f"Overall approval rate: {sub['approved'].mean():.4f}")

for col in ["loan_amount", "income", "property_value", "interest_rate",
            "loan_term", "loan_to_value_ratio"]:
    sub[col] = pd.to_numeric(sub[col], errors="coerce")

def dti_mid(x):
    x = str(x)
    if x in ["NA", "Exempt", ""]:
        return np.nan
    if x == "<20%":
        return 15
    if x == ">60%":
        return 65
    if "-" in x and "%" in x:
        lo, hi = x.replace("%", "").replace("<", "").split("-")
        return (float(lo) + float(hi)) / 2
    try:
        return float(x)
    except ValueError:
        return np.nan

sub["dti_num"] = sub["debt_to_income_ratio"].apply(dti_mid)
sub["loan_amount_k"] = sub["loan_amount"] / 1000
sub["property_value_k"] = sub["property_value"] / 1000
sub["derived_race"] = df.loc[sub.index, "derived_race"]
sub["derived_sex"] = df.loc[sub.index, "derived_sex"]
sub["applicant_age"] = df.loc[sub.index, "applicant_age"]
sub["state_code"] = df.loc[sub.index, "state_code"]

occ_map = {1: "Principal Residence", 2: "Second Residence", 3: "Investment Property"}
sub["occupancy_label"] = sub["occupancy_type"].map(occ_map)

purp_map = {1: "Home Purchase", 2: "Home Improvement", 31: "Refinancing",
            32: "Cash-out Refinancing", 4: "Other Purpose", 5: "Not Applicable"}
sub["purpose_label"] = df.loc[sub.index, "loan_purpose"].map(purp_map)

def dti_bucket(x):
    x = str(x)
    if x in ["NA", "Exempt", ""]:
        return np.nan
    if x == "<20%":
        return "<20%"
    if x == ">60%":
        return ">60%"
    if "-" in x and "%" in x:
        return x
    try:
        v = float(x)
        if v < 20: return "<20%"
        elif v < 30: return "20%-<30%"
        elif v < 36: return "30%-<36%"
        elif v < 50: return "36%-49%"
        elif v <= 60: return "50%-60%"
        else: return ">60%"
    except ValueError:
        return np.nan

sub["dti_bucket"] = sub["debt_to_income_ratio"].apply(dti_bucket)
DTI_ORDER = ["<20%", "20%-<30%", "30%-<36%", "36%-49%", "50%-60%", ">60%"]

ltv_bins = [0, 60, 70, 80, 90, 95, 100, 500]
ltv_labels = ["<=60%", "60-70%", "70-80%", "80-90%", "90-95%", "95-100%", ">100%"]
sub["ltv_bucket"] = pd.cut(sub["loan_to_value_ratio"], bins=ltv_bins, labels=ltv_labels)

inc_bins = [0, 50, 75, 100, 150, 200, 300, 100000]
inc_labels = ["<50K", "50-75K", "75-100K", "100-150K", "150-200K", "200-300K", "300K+"]
sub["income_bucket"] = pd.cut(sub["income"], bins=inc_bins, labels=inc_labels)

# ------------------------------------------------------------------
# 2. CHARTS - APPLICANT PROFILE
# ------------------------------------------------------------------
plt.figure(figsize=(7, 4.5))
sub["applicant_age"].value_counts().reindex(
    ["<25", "25-34", "35-44", "45-54", "55-64", "65-74", ">74"]
).plot(kind="bar", color=COLOR_APPROVED)
plt.title("Applicant Age Distribution, 2025")
plt.ylabel("Number of Applications"); plt.xlabel("Age Bracket")
plt.tight_layout(); plt.savefig(f"{CHART_DIR}/4_3_age_distribution.png"); plt.close()

plt.figure(figsize=(6, 6))
sub["derived_sex"].value_counts().plot(
    kind="pie", autopct="%1.1f%%", colors=sns.color_palette("Blues", 4))
plt.title("Applicant Sex Distribution, 2025"); plt.ylabel("")
plt.tight_layout(); plt.savefig(f"{CHART_DIR}/4_2_sex_distribution.png"); plt.close()

plt.figure(figsize=(8, 4.5))
sub["derived_race"].value_counts().plot(kind="bar", color=COLOR_APPROVED)
plt.title("Applicant Race Distribution, 2025 (Full Applicant Pool)")
plt.ylabel("Number of Applications"); plt.xticks(rotation=45, ha="right")
plt.tight_layout(); plt.savefig(f"{CHART_DIR}/4_3a_race_distribution.png"); plt.close()

plt.figure(figsize=(8, 4.5))
sub["state_code"].value_counts().head(10).plot(kind="bar", color=COLOR_APPROVED)
plt.title("Top 10 States by Application Volume, 2025")
plt.ylabel("Number of Applications"); plt.xlabel("State")
plt.tight_layout(); plt.savefig(f"{CHART_DIR}/4_4_top_states.png"); plt.close()

# ------------------------------------------------------------------
# 3. CHARTS - PRODUCT / OCCUPANCY / PURPOSE APPROVAL RATES
# ------------------------------------------------------------------
def approval_bar(groupcol, title, fname, order=None, figsize=(8, 4.5)):
    rate = sub.groupby(groupcol, observed=True)["approved"].mean().sort_values(ascending=False)
    if order is not None:
        rate = rate.reindex(order)
    plt.figure(figsize=figsize)
    rate.plot(kind="bar", color=COLOR_APPROVED)
    plt.axhline(sub["approved"].mean(), color="red", linestyle="--", linewidth=1,
                label=f"Overall avg ({sub['approved'].mean():.1%})")
    plt.title(title); plt.ylabel("Approval Rate"); plt.legend()
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout(); plt.savefig(f"{CHART_DIR}/{fname}.png"); plt.close()

approval_bar("derived_loan_product_type", "Approval Rate by Loan Product Type", "4_5_product_approval")
approval_bar("occupancy_label", "Approval Rate by Occupancy Type", "4_6_occupancy_approval", figsize=(6, 4.5))
approval_bar("purpose_label", "Approval Rate by Loan Purpose", "4_18_purpose_approval")
approval_bar("dti_bucket", "Approval Rate by Debt-to-Income Bracket", "4_14_dti_approval", order=DTI_ORDER, figsize=(7, 4.5))
approval_bar("ltv_bucket", "Approval Rate by Loan-to-Value Bracket", "4_15_ltv_approval", order=ltv_labels, figsize=(7, 4.5))
approval_bar("derived_race", "Approval Rate by Applicant Race", "4_20_race_approval")

# income bracket - line chart (ordered, monotonic trend)
plt.figure(figsize=(7, 4.5))
sub.groupby("income_bucket", observed=True)["approved"].mean().reindex(inc_labels).plot(
    kind="line", marker="o", color=COLOR_APPROVED, linewidth=2)
plt.title("Approval Rate by Applicant Income Bracket")
plt.ylabel("Approval Rate"); plt.xlabel("Income Bracket")
plt.tight_layout(); plt.savefig(f"{CHART_DIR}/4_17_income_approval.png"); plt.close()

# ------------------------------------------------------------------
# 4. CHARTS - DISTRIBUTIONS
# ------------------------------------------------------------------
plt.figure(figsize=(7, 4.5))
sub["loan_amount"].clip(upper=1_000_000).hist(bins=50, color=COLOR_APPROVED)
plt.title("Distribution of Loan Amount (clipped at $1M for readability)")
plt.xlabel("Loan Amount (USD)"); plt.ylabel("Frequency")
plt.tight_layout(); plt.savefig(f"{CHART_DIR}/4_hist_loan_amount.png"); plt.close()

plt.figure(figsize=(7, 4.5))
sub[(sub["income"] > 0) & (sub["income"] < 1000)]["income"].hist(bins=50, color=COLOR_APPROVED)
plt.title("Distribution of Applicant Income (USD '000s, outliers excluded)")
plt.xlabel("Income (USD '000s)"); plt.ylabel("Frequency")
plt.tight_layout(); plt.savefig(f"{CHART_DIR}/4_hist_income.png"); plt.close()

# box plots: DTI / LTV by approval outcome
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
sns.boxplot(x="approved", y="dti_num", data=sub, ax=axes[0],
            palette=[COLOR_DENIED, COLOR_APPROVED])
axes[0].set_xticklabels(["Denied", "Approved"]); axes[0].set_title("DTI by Approval Outcome")
sns.boxplot(x="approved", y="loan_to_value_ratio", data=sub, ax=axes[1],
            palette=[COLOR_DENIED, COLOR_APPROVED])
axes[1].set_xticklabels(["Denied", "Approved"]); axes[1].set_title("LTV by Approval Outcome")
plt.tight_layout(); plt.savefig(f"{CHART_DIR}/4_8_dti_ltv_boxplots.png"); plt.close()

# scatter: DTI vs LTV colored by approval (sampled for readability)
sample = sub.dropna(subset=["dti_num", "loan_to_value_ratio"]).sample(
    min(20000, len(sub)), random_state=42)
plt.figure(figsize=(7, 6))
plt.scatter(sample["dti_num"], sample["loan_to_value_ratio"],
            c=sample["approved"].map({1: COLOR_APPROVED, 0: "#D9534F"}),
            alpha=0.25, s=8)
plt.xlabel("Debt-to-Income Ratio (%)"); plt.ylabel("Loan-to-Value Ratio (%)")
plt.title("DTI vs LTV, Colored by Approval Outcome (sample of 20,000)")
plt.tight_layout(); plt.savefig(f"{CHART_DIR}/4_scatter_dti_ltv.png"); plt.close()

# correlation heatmap
corr_cols = ["loan_amount", "income", "property_value", "dti_num",
             "loan_to_value_ratio", "interest_rate", "approved"]
plt.figure(figsize=(7, 6))
sns.heatmap(sub[corr_cols].corr(numeric_only=True), annot=True, cmap="RdBu_r",
            center=0, fmt=".2f")
plt.title("Correlation Matrix - Key Numeric Variables")
plt.tight_layout(); plt.savefig(f"{CHART_DIR}/4_25_correlation_heatmap.png"); plt.close()

print("\nDescriptive charts saved to ./charts/")

# ------------------------------------------------------------------
# 5. HYPOTHESIS TESTS (H1-H5)
# ------------------------------------------------------------------
print("\n--- Hypothesis Tests ---")
for col, label in [("dti_num", "H1: DTI"), ("loan_to_value_ratio", "H2: LTV"), ("income", "H3: Income")]:
    a = sub[sub["approved"] == 1][col].dropna()
    d = sub[sub["approved"] == 0][col].dropna()
    t, p = stats.ttest_ind(a, d, equal_var=False)
    print(f"{label}: t={t:.2f}, p={p:.4g}")

ct_product = pd.crosstab(sub["derived_loan_product_type"], sub["approved"])
chi2, p, dof, _ = stats.chi2_contingency(ct_product)
print(f"H4: Product Type: chi2={chi2:.2f}, dof={dof}, p={p:.4g}")

ct_race = pd.crosstab(sub["derived_race"], sub["approved"])
chi2, p, dof, _ = stats.chi2_contingency(ct_race)
print(f"H5: Race: chi2={chi2:.2f}, dof={dof}, p={p:.4g}")

# ------------------------------------------------------------------
# 6. MATCHED-BAND COMPARISON (Section 4.3.7)
# ------------------------------------------------------------------
print("\n--- Matched DTI+LTV Band Comparison (White vs Black) ---")
focus = sub[sub["derived_race"].isin(["White", "Black or African American"]) &
            (sub["dti_bucket"] == "36%-49%") & (sub["ltv_bucket"] == "70-80%")]
print(focus.groupby("derived_race")["approved"].agg(["mean", "count"]))

# ------------------------------------------------------------------
# 7. MODEL DATASET PREP
# ------------------------------------------------------------------
num_feats = ["loan_amount_k", "income", "property_value_k", "loan_term",
             "loan_to_value_ratio", "dti_num"]
cat_feats_baseline = ["derived_loan_product_type", "occupancy_type",
                       "applicant_age", "derived_sex", "loan_purpose"]
cat_feats_expanded = cat_feats_baseline + ["derived_race"]

model_df = sub[num_feats + cat_feats_expanded + ["approved"]].dropna(subset=num_feats).copy()
for c in cat_feats_expanded:
    model_df[c] = model_df[c].astype(str)

print(f"\nFinal modelling dataset: {len(model_df):,} records "
      f"({len(model_df)/len(sub):.1%} of decisioned subset)")

# export cleaned data for Power BI
model_df.to_csv("model_ready_data_powerbi.csv", index=False)
print("Exported model_ready_data_powerbi.csv for Power BI import")

def build_and_evaluate(cat_feats, label):
    X = pd.get_dummies(model_df[num_feats + cat_feats], columns=cat_feats, drop_first=True)
    y = model_df["approved"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y)

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    logit = LogisticRegression(max_iter=1000)
    logit.fit(X_train_s, y_train)
    pred_l = logit.predict(X_test_s)
    proba_l = logit.predict_proba(X_test_s)[:, 1]

    rf = RandomForestClassifier(n_estimators=300, max_depth=14, min_samples_leaf=5,
                                 random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    pred_r = rf.predict(X_test)
    proba_r = rf.predict_proba(X_test)[:, 1]

    print(f"\n=== {label} ===")
    for name, pred, proba in [("Logistic Regression", pred_l, proba_l), ("Random Forest", pred_r, proba_r)]:
        print(f"{name}: Acc={accuracy_score(y_test,pred):.4f} "
              f"Prec={precision_score(y_test,pred):.4f} Rec={recall_score(y_test,pred):.4f} "
              f"F1={f1_score(y_test,pred):.4f} AUC={roc_auc_score(y_test,proba):.4f}")

    importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
    print("Top 10 features:\n", importances.head(10))

    return dict(X_test=X_test, y_test=y_test, pred_l=pred_l, proba_l=proba_l,
                pred_r=pred_r, proba_r=proba_r, importances=importances, rf=rf)

baseline = build_and_evaluate(cat_feats_baseline, "BASELINE MODEL (financial + loan variables only)")
expanded = build_and_evaluate(cat_feats_expanded, "EXPANDED MODEL (+ demographic variables)")

# ------------------------------------------------------------------
# 8. MODEL CHARTS
# ------------------------------------------------------------------
# ROC curve comparison (baseline RF vs expanded RF)
plt.figure(figsize=(6.5, 6))
for label, res, color in [("Baseline RF", baseline, "#2E5C8A"), ("Expanded RF", expanded, "#5CB85C")]:
    fpr, tpr, _ = roc_curve(res["y_test"], res["proba_r"])
    auc = roc_auc_score(res["y_test"], res["proba_r"])
    plt.plot(fpr, tpr, label=f"{label} (AUC={auc:.3f})", color=color, linewidth=2)
plt.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random guess")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve: Random Forest, Baseline vs Expanded Model")
plt.legend(); plt.tight_layout(); plt.savefig(f"{CHART_DIR}/4_roc_curve.png"); plt.close()

# Feature importance chart (expanded model, top 15)
plt.figure(figsize=(8, 6))
expanded["importances"].head(15).sort_values().plot(kind="barh", color=COLOR_APPROVED)
plt.title("Top 15 Feature Importances\nExpanded Random Forest Model")
plt.xlabel("Relative Importance")
plt.tight_layout(); plt.savefig(f"{CHART_DIR}/4_22_feature_importance.png"); plt.close()

# Confusion matrix heatmap (expanded RF)
cm = confusion_matrix(expanded["y_test"], expanded["pred_r"])
plt.figure(figsize=(5.5, 5))
sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues",
            xticklabels=["Denied", "Approved"], yticklabels=["Denied", "Approved"])
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("Confusion Matrix - Expanded Random Forest Model")
plt.tight_layout(); plt.savefig(f"{CHART_DIR}/4_confusion_matrix.png"); plt.close()

print("\nModel charts saved to ./charts/")
print("\nAll done.")

Raw records: 663,472
Decisioned records (Originated/Denied): 539,966
Overall approval rate: 0.7951


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_18912\246810693.py:190: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x="approved", y="dti_num", data=sub, ax=axes[0],
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_18912\246810693.py:192: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axes[0].set_xticklabels(["Denied", "Approved"]); axes[0].set_title("DTI by Approval Outcome")
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_18912\246810693.py:193: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x="approved", y="loan_to_value_ratio", data=sub, ax=axes[1],
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_18912\246810693.py:195: 


Descriptive charts saved to ./charts/

--- Hypothesis Tests ---
H1: DTI: t=-107.83, p=0
H2: LTV: t=-35.96, p=4.464e-282
H3: Income: t=43.06, p=0
H4: Product Type: chi2=7280.21, dof=4, p=0
H5: Race: chi2=2448.79, dof=7, p=0

--- Matched DTI+LTV Band Comparison (White vs Black) ---
                               mean  count
derived_race                              
Black or African American  0.770723   6961
White                      0.849091  53211

Final modelling dataset: 514,253 records (95.2% of decisioned subset)
Exported model_ready_data_powerbi.csv for Power BI import

=== BASELINE MODEL (financial + loan variables only) ===
Logistic Regression: Acc=0.8021 Prec=0.8061 Rec=0.9885 F1=0.8880 AUC=0.6907
Random Forest: Acc=0.8549 Prec=0.8475 Rec=0.9966 F1=0.9160 AUC=0.7847
Top 10 features:
 dti_num                                                    0.394937
loan_to_value_ratio                                        0.207271
income                                                     

In [2]:
import matplotlib.cm as cm
import matplotlib.colors as mcolors
 
STATE_GRID = {
 "AK": (0,0), "ME": (0,10),
 "VT": (1,9), "NH": (1,10),
 "WA": (1,0), "ID": (1,1), "MT": (1,2), "ND": (1,3), "MN": (1,4), "WI": (1,5), "MI": (1,6),
 "NY": (1,8), "MA": (2,10), "RI": (2,11),
 "OR": (2,0), "NV": (2,1), "WY": (2,2), "SD": (2,3), "IA": (2,4), "IL": (2,5), "IN": (2,6),
 "OH": (2,7), "PA": (2,8), "NJ": (2,9), "CT": (3,10),
 "CA": (3,0), "UT": (3,1), "CO": (3,2), "NE": (3,3), "MO": (3,4), "KY": (3,5), "WV": (3,6),
 "VA": (3,7), "MD": (3,8), "DE": (3,9),
 "AZ": (4,1), "NM": (4,2), "KS": (4,3), "AR": (4,4), "TN": (4,5), "NC": (4,6), "SC": (4,7), "DC": (4,8),
 "OK": (5,3), "LA": (5,4), "MS": (5,5), "AL": (5,6), "GA": (5,7),
 "HI": (6,0), "TX": (6,3), "FL": (6,8),
}
_MAX_ROW = max(r for r, c in STATE_GRID.values()) + 1
_MAX_COL = max(c for r, c in STATE_GRID.values()) + 1
 
def draw_state_tile_map(values, title, cbar_label, cmap_name, fname, is_pct=False):
    fig, ax = plt.subplots(figsize=(13, 7.5))
    present = {s: v for s, v in values.items() if s in STATE_GRID}
    vmin, vmax = min(present.values()), max(present.values())
    cmap = plt.get_cmap(cmap_name)
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    for state, (r, c) in STATE_GRID.items():
        v = values.get(state, np.nan)
        color = cmap(norm(v)) if not np.isnan(v) else "#e8e8e8"
        ax.add_patch(plt.Rectangle((c, _MAX_ROW - 1 - r), 0.92, 0.92,
                                    facecolor=color, edgecolor="white", linewidth=1.5))
        light_text = (not np.isnan(v)) and norm(v) > 0.55
        ax.text(c + 0.46, _MAX_ROW - 1 - r + 0.58, state, ha="center", va="center",
                fontsize=10, fontweight="bold", color="white" if light_text else "#333333")
        if not np.isnan(v):
            label = f"{v:.0%}" if is_pct else f"{v:,.0f}"
            ax.text(c + 0.46, _MAX_ROW - 1 - r + 0.28, label, ha="center", va="center",
                    fontsize=7.5, color="white" if light_text else "#333333")
    ax.set_xlim(-0.2, _MAX_COL + 0.2); ax.set_ylim(-0.2, _MAX_ROW + 0.2)
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(title, fontsize=16, fontweight="bold", pad=14)
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label(cbar_label, fontsize=10)
    plt.tight_layout(); plt.savefig(fname, dpi=150, bbox_inches="tight"); plt.close()
 
draw_state_tile_map(sub["state_code"].value_counts().to_dict(),
                     "Rocket Mortgage 2025 Application Volume by State", "Applications", "Blues",
                     f"{CHART_DIR}/4_map_application_volume_by_state.png")
draw_state_tile_map(sub.groupby("state_code")["approved"].mean().to_dict(),
                     "Rocket Mortgage 2025 Approval Rate by State", "Approval Rate", "RdYlGn",
                     f"{CHART_DIR}/4_19_map_approval_rate_by_state.png", is_pct=True)
 
print("\nDescriptive charts (incl. state maps) saved to ./charts/")


Descriptive charts (incl. state maps) saved to ./charts/
